# **IBM Data Science Professional Certificate Capstone Project**

## **Launch Sites Locations Analysis with Folium**


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities to a launch site. Finding an optimal location for building a launch site certainly involves many factors that we can discover by analyzing the launch site locations.

In the previous EDA, we visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. Here, we will perform interactive visual analytics using `Folium`.

### Install packages and import libraries

In [21]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [22]:
import folium
import pandas as pd

In [23]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

### Step 1

First, let's add each site's location on a map using its latitude and longitude coordinates. The following dataset (`spacex_launch_geo.csv`) is an augmented dataset with latitude and longitude added for each site.

In [24]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

We can take a look at what are the coordinates of each site.

In [26]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


The above coordinates are just plain numbers that can not give us any intuitive insights about where are those launch sites located. We will visualize those locations by pinning them on a map.
First, we need to create a folium `Map` object with an initial location. We choose the NASA Johnson Space Center in Houston, Texas as our initial location.

In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We use `folium.Circle` to add a highlighted circle area with a text label on the specific coordinates.

In [ ]:
# Create a circle at the NASA Johnson Space Center's coordinates with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a marker at the NASA Johnson Space Center's coordinates 
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Now, let's add a circle for each launch site in data frame `launch_sites`.

In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Create a circle for each launch site using their coordinates and add a popup label showing their names
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(row["Launch Site"])).add_to(site_map)
    folium.map.Marker(coordinate,
        # Create an icon as a text label
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row["Launch Site"],
            )
        ).add_to(site_map)

site_map


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


The launch sites in Florida are closer to the Equator line than the launch site in Los Angeles. 
The Florida launch sites are close to the Atlantic Ocean while the launch site in Los Angeles is close to the Pacific Ocean.

### Step 2

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates. The data frame spacex_df has detailed launch records, and the `class` column indicates if the launch was successful or not.


In [30]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()

Create a new column in the dataframe called `marker_color` to store the marker colors based on the `class` value.

In [ ]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

def launch_site_color(result):
   if result==1:
       return "green"
   else:
       return "red"
        
spacex_df["marker_color"]=spacex_df["class"].apply(launch_site_color)

spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


For each launch result add a `folium.Marker` to `marker_cluster.`

In [ ]:
# Add marker_cluster to current site_map
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame create a Marker object with its coordinate and customize the Marker's icon property
# to indicate if this launch was successed or failed, e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.map.Marker(coordinate,
                icon=folium.Icon(color="white", icon_color=row["marker_color"])
                ).add_to(marker_cluster)
    
site_map

# Note: The iterrows() method generates an iterator object of the DataFrame, allowing us to iterate through each row in the dataFrame. 
# Each iteration produces an index object and a row object (a Pandas Series object).

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, we are able to easily identify which launch sites have relatively high success rates. KSC LC-39A had the best success rate, out of 13 launches, 10 were successful. 

### Step 3

Next, we need to explore and analyze the proximities of launch sites. Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while we are exploring the map, we can easily find the coordinates of any points of interests (such as railways)

In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now when zooming into a launch site, we can explore its proximity to see if we can find any railway, highway, coastline, etc. When moving the mouse to these points, we mark down their coordinates (shown on the top-left) in order to calculate its distance to the launch site.

In [ ]:
# Calculate the distance between a launch site to its proximities
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

We markdown a point on the closest coastline using the mouse position and calculate the distance between the coastline point and the launch site.

In [ ]:
# Coordinates for closest coast line: Lat: 28.5634, Long: -80.56807
# Coordinates of launch site CCAFS SLC-40: Lat:28.563197, Long:	-80.576820

launch_site_lat=28.563197
launch_site_lon=-80.576820
coastline_lat=28.5634
coastline_lon=-80.56807

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

0.8551030503271537

In [ ]:
# Create and add a folium.Marker on your selected closest coastline point on the map
distance_marker = folium.Marker(
   [coastline_lat, coastline_lon], 
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
       )
   )

site_map.add_child(distance_marker)
site_map

We draw a `PolyLine` between a launch site to the selected coastline point.

In [ ]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinates
coordinates=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


Similarly, we can draw a line betwee a launch site to its closest city, railway, highway, etc. We need to use `MousePosition` to find the their coordinates on the map first.


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Calculate the distance between the launch site and the closest city (Titusville)
launch_site_lat=28.563197
launch_site_lon=-80.576820
city_lat=28.61222
city_lon=-80.80796

distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

distance_marker = folium.Marker(
   [city_lat, city_lon], 
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),
       )
   )


site_map.add_child(distance_marker)


coordinates=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

site_map

In [ ]:
# Calculate the distance between the launch site and the closest highway
launch_site_lat=28.563197
launch_site_lon=-80.576820
highway_lat=28.56377
highway_lon=-80.57083

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)

distance_marker = folium.Marker(
   [highway_lat, highway_lon], 
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway),
       )
   )


site_map.add_child(distance_marker)


coordinates=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

site_map

In [ ]:
# Calculate the distance between the launch site and the closest railroad
launch_site_lat=28.563197
launch_site_lon=-80.576820
railroad_lat=28.5712
railroad_lon=-80.58536

distance_railroad = calculate_distance(launch_site_lat, launch_site_lon, railroad_lat, railroad_lon)

distance_marker = folium.Marker(
   [railroad_lat, railroad_lon], 
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railroad),
       )
   )


site_map.add_child(distance_marker)


coordinates=[[launch_site_lat, launch_site_lon], [railroad_lat, railroad_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

site_map